# Hook 机制

`Agent` 在工具执行前后以及每轮结束时提供多个 Hook：

- `before_tool_call`：拦截/放行工具调用
- `after_tool_call`：修改工具结果
- `should_stop_after_turn`：决定是否结束当前 run
- `prepare_next_turn`：修改下一轮上下文/模型/思考级别


In [1]:
import os

os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"


In [2]:
from nova_agent import (
    Agent, AgentTool, AgentToolResult,
    BeforeToolCallContext, BeforeToolCallResult,
    AfterToolCallContext, AfterToolCallResult,
    ShouldStopAfterTurnContext,
    PrepareNextTurnContext, AgentLoopTurnUpdate,
)
from nova_ai import TextContent

class DangerousTool(AgentTool[dict, dict]):
    name: str = "exec_shell"
    description: str = "执行 shell 命令（示例中被禁止）"
    parameters: dict = {
        "type": "object",
        "properties": {"cmd": {"type": "string"}},
        "required": ["cmd"],
    }
    label: str = "shell"

    async def execute(self, tool_call_id, params, signal=None, on_update=None):
        return AgentToolResult(
            content=[TextContent(text="这不应该被执行")],
            details={},
        )

class EchoTool(AgentTool[dict, dict]):
    name: str = "echo"
    description: str = "原样返回输入"
    parameters: dict = {
        "type": "object",
        "properties": {"text": {"type": "string"}},
        "required": ["text"],
    }
    label: str = "回显"

    async def execute(self, tool_call_id, params, signal=None, on_update=None):
        return AgentToolResult(
            content=[TextContent(text=f"echo: {params.get('text')}")],
            details=params,
        )

def before_tool(ctx: BeforeToolCallContext, signal=None):
    if ctx.tool_call.name == "exec_shell":
        print(f"阻止危险工具: {ctx.tool_call.name}")
        return BeforeToolCallResult(block=True, reason="安全策略禁止执行 shell")
    return None

def after_tool(ctx: AfterToolCallContext, signal=None):
    # 对 echo 结果追加前缀
    if ctx.tool_call.name == "echo" and not ctx.is_error:
        new_text = "[经过 after_hook] " + ctx.result.content[0].text
        return AfterToolCallResult(
            content=[TextContent(text=new_text)],
            details=ctx.result.details,
        )
    return None

def should_stop(ctx: ShouldStopAfterTurnContext):
    # 当 assistant 消息中出现 "结束" 时停止
    text = ""
    for c in ctx.message.content:
        if getattr(c, "type", None) == "text":
            text += c.text
    if "结束" in text:
        print("\n[should_stop_after_turn] 检测到结束词，停止 run")
        return True
    return False

def prepare_next(ctx: PrepareNextTurnContext):
    # 这里可以动态替换上下文或模型；示例仅打印
    print(f"[prepare_next_turn] 当前 turn 新增 {len(ctx.new_messages)} 条消息")
    return None

agent = Agent(
    initial_state={"system_prompt": "你可以调用 echo 工具，永远不要调用 exec_shell。"},
    before_tool_call=before_tool,
    after_tool_call=after_tool,
    should_stop_after_turn=should_stop,
    prepare_next_turn=prepare_next,
)
agent.set_tools([DangerousTool(), EchoTool()])

await agent.prompt("请用 echo 工具说一句话，并在结尾包含'结束'二字。")
await agent.wait_for_idle()
print("\n最终消息:", agent.state.messages[-1].content[0].text)


[prepare_next_turn] 当前 turn 新增 3 条消息
[prepare_next_turn] 当前 turn 新增 4 条消息

[should_stop_after_turn] 检测到结束词，停止 run

最终消息: 已按照您的要求，使用 echo 工具说了一句话，并在结尾包含了"结束"二字。
